In [1]:
import pandas as pd
import numpy as np
import ast

from pathlib import Path
from datetime import datetime, timedelta
import dateparser

In [2]:
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
merged_events = pd.read_csv(
    DATA_DIR / "merged_events.csv"
)

print("Total consolidated events:", len(merged_events))

merged_events.head()

Total consolidated events: 48


,cluster_id,representative_sentence,article_ids,source_sentences,event_triggers,actions,subjects,objects,temporal_entities,location_entities,negation,event_count,article_count
0,1,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['began'],['begin'],['Continuous rainfall'],[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1
1,2,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['causing'],['cause'],['Continuous rainfall'],[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1
2,3,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['flood'],['flood'],['several streets in the eastern part of the c...,[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1
3,4,The city weather office recorded 118 millimetr...,['article_01'],['The city weather office recorded 118 millime...,['recorded'],['record'],['The city weather office'],['118 millimetres of rain'],"[""['twelve hours']""]",['[]'],False,1,1
4,5,Emergency services received their first major ...,['article_01'],['Emergency services received their first majo...,['received'],['receive'],['Emergency services'],['their first major flooding calls'],"[""['approximately 5:30 a.m.', 'August 14']""]",['[]'],False,1,1


In [4]:
extracted_events = pd.read_csv(
    DATA_DIR / "extracted_events.csv"
)

print("Original extracted events:", len(extracted_events))

print("Available columns:")
print(extracted_events.columns.tolist())

Original extracted events: 52
Available columns:
['article_id', 'sentence_id', 'publication_date', 'original_sentence', 'event_trigger', 'action', 'subject', 'object', 'agent', 'passive', 'negation', 'temporal_entities', 'location_entities']


In [6]:
# Create a mapping between each article and its publication date
article_dates = (
    extracted_events[["article_id", "publication_date"]]
    .dropna(subset=["publication_date"])
    .drop_duplicates(subset=["article_id"])
)

article_date_map = dict(
    zip(article_dates["article_id"], article_dates["publication_date"])
)

# Parse article_ids stored as lists in merged_events.csv
def get_publication_dates(article_ids):
    if pd.isna(article_ids):
        return []

    try:
        ids = ast.literal_eval(article_ids)
    except:
        ids = [article_ids]

    return list(dict.fromkeys(
        article_date_map[article_id]
        for article_id in ids
        if article_id in article_date_map
    ))

merged_events["publication_dates"] = (
    merged_events["article_ids"].apply(get_publication_dates)
)

merged_events[["cluster_id", "article_ids", "publication_dates"]].head()

,cluster_id,article_ids,publication_dates
0,1,['article_01'],[2026-08-14]
1,2,['article_01'],[2026-08-14]
2,3,['article_01'],[2026-08-14]
3,4,['article_01'],[2026-08-14]
4,5,['article_01'],[2026-08-14]


In [7]:
print("MERGED EVENTS")
display(merged_events.head())

print("\nEXTRACTED EVENTS")
display(extracted_events.head())

MERGED EVENTS


,cluster_id,representative_sentence,article_ids,source_sentences,event_triggers,actions,subjects,objects,temporal_entities,location_entities,negation,event_count,article_count,publication_dates
0,1,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['began'],['begin'],['Continuous rainfall'],[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1,[2026-08-14]
1,2,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['causing'],['cause'],['Continuous rainfall'],[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1,[2026-08-14]
2,3,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['flood'],['flood'],['several streets in the eastern part of the c...,[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1,[2026-08-14]
3,4,The city weather office recorded 118 millimetr...,['article_01'],['The city weather office recorded 118 millime...,['recorded'],['record'],['The city weather office'],['118 millimetres of rain'],"[""['twelve hours']""]",['[]'],False,1,1,[2026-08-14]
4,5,Emergency services received their first major ...,['article_01'],['Emergency services received their first majo...,['received'],['receive'],['Emergency services'],['their first major flooding calls'],"[""['approximately 5:30 a.m.', 'August 14']""]",['[]'],False,1,1,[2026-08-14]



EXTRACTED EVENTS


,article_id,sentence_id,publication_date,original_sentence,event_trigger,action,subject,object,agent,passive,negation,temporal_entities,location_entities
0,article_01,1,2026-08-14,Continuous rainfall began across Riverton late...,began,begin,Continuous rainfall,NaN,NaN,False,False,"['August 13', 'early Friday', 'morning']","['Riverton', 'the eastern part of the city']"
1,article_01,1,2026-08-14,Continuous rainfall began across Riverton late...,causing,cause,Continuous rainfall,NaN,NaN,False,False,"['August 13', 'early Friday', 'morning']","['Riverton', 'the eastern part of the city']"
2,article_01,1,2026-08-14,Continuous rainfall began across Riverton late...,flood,flood,several streets in the eastern part of the city,NaN,NaN,False,False,"['August 13', 'early Friday', 'morning']","['Riverton', 'the eastern part of the city']"
3,article_01,2,2026-08-14,The city weather office recorded 118 millimetr...,recorded,record,The city weather office,118 millimetres of rain,NaN,False,False,['twelve hours'],[]
4,article_01,3,2026-08-14,Emergency services received their first major ...,received,receive,Emergency services,their first major flooding calls,NaN,False,False,"['approximately 5:30 a.m.', 'August 14']",[]


In [8]:
timeline_df = merged_events.copy()

# Keep relevant columns
timeline_df = timeline_df[
    [
        'cluster_id',
        'representative_sentence',
        'article_ids',
        'source_sentences',
        'event_triggers',
        'actions',
        'subjects',
        'objects',
        'temporal_entities',
        'location_entities',
        'negation',
        'event_count',
        'article_count',
        'publication_dates'
    ]
].copy()

print("Timeline records:", len(timeline_df))
display(timeline_df.head())

Timeline records: 48


,cluster_id,representative_sentence,article_ids,source_sentences,event_triggers,actions,subjects,objects,temporal_entities,location_entities,negation,event_count,article_count,publication_dates
0,1,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['began'],['begin'],['Continuous rainfall'],[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1,[2026-08-14]
1,2,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['causing'],['cause'],['Continuous rainfall'],[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1,[2026-08-14]
2,3,Continuous rainfall began across Riverton late...,['article_01'],['Continuous rainfall began across Riverton la...,['flood'],['flood'],['several streets in the eastern part of the c...,[],"[""['August 13', 'early Friday', 'morning']""]","[""['Riverton', 'the eastern part of the city']""]",False,1,1,[2026-08-14]
3,4,The city weather office recorded 118 millimetr...,['article_01'],['The city weather office recorded 118 millime...,['recorded'],['record'],['The city weather office'],['118 millimetres of rain'],"[""['twelve hours']""]",['[]'],False,1,1,[2026-08-14]
4,5,Emergency services received their first major ...,['article_01'],['Emergency services received their first majo...,['received'],['receive'],['Emergency services'],['their first major flooding calls'],"[""['approximately 5:30 a.m.', 'August 14']""]",['[]'],False,1,1,[2026-08-14]


In [9]:
import ast

def parse_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except (ValueError, SyntaxError):
        return [value]


list_columns = [
    'article_ids',
    'source_sentences',
    'event_triggers',
    'actions',
    'subjects',
    'objects',
    'temporal_entities',
    'location_entities',
    'publication_dates'
]

for col in list_columns:
    timeline_df[col] = timeline_df[col].apply(parse_list)

display(timeline_df[
    ['cluster_id', 'temporal_entities', 'publication_dates', 'location_entities']
].head())

,cluster_id,temporal_entities,publication_dates,location_entities
0,1,"[['August 13', 'early Friday', 'morning']]",[2026-08-14],"[['Riverton', 'the eastern part of the city']]"
1,2,"[['August 13', 'early Friday', 'morning']]",[2026-08-14],"[['Riverton', 'the eastern part of the city']]"
2,3,"[['August 13', 'early Friday', 'morning']]",[2026-08-14],"[['Riverton', 'the eastern part of the city']]"
3,4,[['twelve hours']],[2026-08-14],[[]]
4,5,"[['approximately 5:30 a.m.', 'August 14']]",[2026-08-14],[[]]


In [10]:
timeline_df['publication_dates'] = timeline_df['publication_dates'].apply(
    lambda dates: [
        pd.to_datetime(date, errors='coerce')
        for date in dates
    ]
)

def get_publication_date(dates):
    valid_dates = [d for d in dates if pd.notna(d)]
    return min(valid_dates) if valid_dates else pd.NaT

timeline_df['publication_date'] = timeline_df['publication_dates'].apply(
    get_publication_date
)

display(timeline_df[
    ['cluster_id', 'publication_dates', 'publication_date']
].head(10))

,cluster_id,publication_dates,publication_date
0,1,[2026-08-14 00:00:00],2026-08-14
1,2,[2026-08-14 00:00:00],2026-08-14
2,3,[2026-08-14 00:00:00],2026-08-14
3,4,[2026-08-14 00:00:00],2026-08-14
4,5,[2026-08-14 00:00:00],2026-08-14
5,6,[2026-08-14 00:00:00],2026-08-14
6,7,[2026-08-14 00:00:00],2026-08-14
7,8,"[2026-08-14 00:00:00, 2026-08-15 00:00:00]",2026-08-14
8,9,[2026-08-14 00:00:00],2026-08-14
9,10,[2026-08-14 00:00:00],2026-08-14


In [11]:
def extract_event_date(temporal_entities, publication_date):

    if pd.isna(publication_date):
        return pd.NaT

    extracted_dates = []

    for expression in temporal_entities:

        parsed_date = dateparser.parse(
            expression,
            settings={
                'RELATIVE_BASE': publication_date.to_pydatetime(),
                'PREFER_DATES_FROM': 'past'
            }
        )

        if parsed_date:
            extracted_dates.append(parsed_date.date())

    if extracted_dates:
        return min(extracted_dates)

    return publication_date.date()

In [17]:
timeline_df['event_date'] = timeline_df.apply(
    lambda row: extract_event_date(
        row['temporal_entities'],
        row['publication_date']
    ),
    axis=1
)

display(timeline_df[
    ['cluster_id', 'temporal_entities', 'publication_date', 'event_date']
].head(15))


,cluster_id,temporal_entities,publication_date,event_date
0,34,"[['twelve hours', 'Friday', 'morning']]",2026-08-15,2025-12-15
1,35,"[['twelve hours', 'Friday', 'morning']]",2026-08-15,2025-12-15
2,36,"[['twelve hours', 'Friday', 'morning']]",2026-08-15,2025-12-15
3,11,[['Saturday']],2026-08-14,2026-08-08
4,12,[['Saturday']],2026-08-14,2026-08-08
5,29,"[['Saturday', 'evening']]",2026-08-14,2026-08-08
6,48,"[['Saturday', 'afternoon']]",2026-08-15,2026-08-08
7,28,[['4 p.m.']],2026-08-14,2026-08-13
8,27,[['4 p.m.']],2026-08-14,2026-08-13
9,25,[['noon']],2026-08-14,2026-08-13


In [20]:
timeline_df = timeline_df.sort_values(
    by=['event_date'],
    na_position='last'
).reset_index(drop=True)

display(timeline_df[
    ['cluster_id', 'event_date', 'representative_sentence',
     'location_entities', 'article_count']
].head(20))

,cluster_id,event_date,representative_sentence,location_entities,article_count
0,34,2025-12-15,The weather office reported that the city had ...,[[]],1
1,35,2025-12-15,The weather office reported that the city had ...,[[]],1
2,36,2025-12-15,The weather office reported that the city had ...,[[]],1
3,11,2026-08-08,The weather office warned that further rainfal...,[[]],1
4,12,2026-08-08,The weather office warned that further rainfal...,[[]],1
5,29,2026-08-08,The weather office forecast more rain until Sa...,[[]],1
6,48,2026-08-08,No deaths had been reported as of Saturday aft...,[[]],1
7,25,2026-08-13,"By noon, approximately 60 people were staying ...",[['the two locations']],1
8,28,2026-08-13,The district collector said that a review meet...,[[]],1
9,9,2026-08-13,Officials said that 46 residents had been move...,[[]],1


In [14]:
timeline = timeline_df[
    [
        'cluster_id',
        'event_date',
        'representative_sentence',
        'location_entities',
        'article_ids',
        'article_count'
    ]
].copy()

timeline.rename(columns={
    'cluster_id': 'event_id',
    'event_date': 'date',
    'representative_sentence': 'event',
    'location_entities': 'locations',
    'article_ids': 'related_articles',
    'article_count': 'reported_by'
}, inplace=True)

display(timeline)

/var/folders/p1/z_8f7sw554s8tqj8wmqcx2y00000gn/T/ipykernel_14583/4084571119.py:12: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  timeline.rename(columns={


,event_id,date,event,locations,related_articles,reported_by
0,34,2025-12-15,The weather office reported that the city had ...,[[]],[article_03],1
1,35,2025-12-15,The weather office reported that the city had ...,[[]],[article_03],1
2,36,2025-12-15,The weather office reported that the city had ...,[[]],[article_03],1
3,11,2026-08-08,The weather office warned that further rainfal...,[[]],[article_01],1
4,12,2026-08-08,The weather office warned that further rainfal...,[[]],[article_01],1
5,29,2026-08-08,The weather office forecast more rain until Sa...,[[]],[article_02],1
6,48,2026-08-08,No deaths had been reported as of Saturday aft...,[[]],[article_03],1
7,4,2026-08-13,The city weather office recorded 118 millimetr...,[[]],[article_01],1
8,9,2026-08-13,Officials said that 46 residents had been move...,[[]],[article_01],1
9,10,2026-08-13,Officials said that 46 residents had been move...,[[]],[article_01],1


In [19]:
output_file = "../outputs/final_news_timeline_v1.txt"

with open(output_file, "w", encoding="utf-8") as f:

    f.write("=" * 90 + "\n")
    f.write("                  NEWS EVENT TIMELINE\n")
    f.write("=" * 90 + "\n\n")

    for _, row in timeline.iterrows():

        f.write(f"DATE: {row['date']}\n")
        f.write(f"EVENT ID: {row['event_id']}\n")
        f.write(f"EVENT: {row['event']}\n")
        f.write(f"LOCATION: {', '.join(row['locations']) if row['locations'] else 'Unknown'}\n")
        f.write(f"REPORTED BY: {row['reported_by']} article(s)\n")
        f.write(f"RELATED ARTICLES: {', '.join(row['related_articles'])}\n")

        f.write("-" * 90 + "\n\n")

print(f"Timeline successfully saved to {output_file}")

Timeline successfully saved to ../outputs/final_news_timeline_v1.txt
